<a href="https://colab.research.google.com/github/a-forty-two/vodafonecairo31aug/blob/main/Model_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mastering LLM Deployment
## Lab 3 - Model Quantization

**Duration:** ~90 minutes  ·  **Runtime:** T4 GPU  ·  **Prerequisite:** Lab 1 (`models/teacher-bert-sst2/` must exist)

---

### Why this lab matters most

In the Lab 1 case study, quantization was reported as the single largest contributor to the roughly 30× improvement, at a cost of under one percentage point of F1. Of the three Day 1 levers it has the best return-to-risk ratio, for a simple reason: **it does not change what the model computes, only how precisely it represents the numbers.** No retraining is required, so the failure mode is a measurable accuracy drop rather than an unpredictable behaviour change.

### The two things this lab measures separately

There is a trap here that costs teams weeks, so we address it head-on. Quantization has two distinct effects, and they are measured with different tools:

| Effect | What it needs | What we use |
|---|---|---|
| **Accuracy impact** | quantize-then-dequantize the weights and re-evaluate | simulated quantization, applied to the real BERT model |
| **Size and speed impact** | a runtime that actually stores and multiplies int8 | TFLite, on a model it converts cleanly |

Simulating quantization on a large model tells you honestly whether quality survives; it will *not* make the model smaller or faster, because the tensors are still fp32 containers holding rounded values. Conversely, a TFLite benchmark gives you real bytes and real milliseconds. **This lab does both, on the model where each is appropriate,** and is explicit about which number came from which.

### Learning outcomes

- Derive and implement affine quantization from scratch; explain scale, zero-point, and clipping error.
- Compare per-tensor and per-channel quantization and predict when the difference matters.
- Distinguish dynamic-range, float16, and full-integer post-training quantization, and select a calibration set.
- Apply post-training quantization with TFLite and quantization-aware training with `tensorflow_model_optimization`, and measure real size, latency and accuracy on CPU.
- Quantize the Lab 1 BERT classifier and test it under distribution shift (SST-2 fragments → IMDB reviews).
- Add quantization rows to the optimization ledger.

**Expected GPU/CPU time: 10–14 minutes.**

---
## 0. Environment setup

In [ ]:
%pip install -q "transformers>=4.40,<5" "datasets>=2.19,<4" "tf-keras>=2.16" "tensorflow-model-optimization>=0.8.0" "scikit-learn" "pandas"
print('dependencies installed')

In [ ]:
import os, sys
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
if "tensorflow" in sys.modules:
    print("TensorFlow already imported -> Runtime > Restart session and re-run from the top.")

import tensorflow as tf, numpy as np, time, transformers
# tf.keras is a lazy loader and does not re-export __version__.
try:
    import tf_keras as _keras_pkg
except ImportError:
    import keras as _keras_pkg
_keras_impl = tf.keras.Model.__module__
print("TF", tf.__version__, "| Keras", _keras_pkg.__version__, "|", _keras_impl)
assert _keras_pkg.__version__.startswith("2.") and "tf_keras" in _keras_impl, (
    "Keras 2 is not active. Install tf-keras, set TF_USE_LEGACY_KERAS=1 before "
    "importing tensorflow, then Runtime > Restart session.")
tf.keras.utils.set_random_seed(42)

In [ ]:
USE_DRIVE = True
ROOT = "/content/llm-deploy-labs"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        ROOT = "/content/drive/MyDrive/llm-deploy-labs"
    except Exception as e:
        print("Drive unavailable:", e)
os.environ["LLMDEPLOY_ROOT"] = ROOT
for sub in ("models", "reports", "data"):
    os.makedirs(os.path.join(ROOT, sub), exist_ok=True)
print("Artifact root:", ROOT)

In [ ]:
# Same lab kit as Labs 1-2, rewritten so this notebook stands alone.
LABKIT_SRC = r'''
"""
labkit.py - shared utilities for the "Mastering LLM Deployment" hands-on labs.

Everything the labs need in common lives here so that each notebook measures
the same things in the same way:

  * artifact + ledger management (results survive across notebooks via Drive)
  * a model "size on disk" and parameter/sparsity accounting
  * a latency/throughput benchmark harness with warm-up and percentiles
  * a minimal, explicit GradientTape training loop (works for HF TF models,
    plain Keras models, distillation losses and masked/pruned training alike)
"""

import os

os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")

import json
import shutil
import time
from pathlib import Path

import numpy as np
import tensorflow as tf

# --------------------------------------------------------------------------
# 1. Artifact root
# --------------------------------------------------------------------------

_ROOT = Path(os.environ.get("LLMDEPLOY_ROOT", "/content/llm-deploy-labs"))


def set_root(path):
    """Point the lab kit at a persistent directory (ideally on Google Drive)."""
    global _ROOT
    _ROOT = Path(path)
    for sub in ("models", "reports", "data"):
        (_ROOT / sub).mkdir(parents=True, exist_ok=True)
    os.environ["LLMDEPLOY_ROOT"] = str(_ROOT)
    return _ROOT


def root():
    return _ROOT


def model_dir(name, clean=False):
    """Return (and create) a directory under <root>/models/<name>."""
    d = _ROOT / "models" / name
    if clean and d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    return d


# --------------------------------------------------------------------------
# 2. Size and parameter accounting
# --------------------------------------------------------------------------


def size_mb(path):
    """Size of a file or, recursively, of a directory - in MB."""
    p = Path(path)
    if p.is_file():
        return p.stat().st_size / 1e6
    total = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
    return total / 1e6


def count_params(model):
    """Total trainable parameter count."""
    return int(sum(int(np.prod(v.shape)) for v in model.trainable_variables))


def weight_sparsity(model, kinds=("kernel", "weight", "embeddings")):
    """Fraction of zeros across the "real" weight matrices (ignores biases /
    LayerNorm, which are never pruned in practice)."""
    zeros, total = 0, 0
    for v in model.trainable_variables:
        if not any(k in v.name for k in kinds):
            continue
        arr = v.numpy()
        zeros += int((arr == 0).sum())
        total += int(arr.size)
    return zeros / max(total, 1)


# --------------------------------------------------------------------------
# 3. Latency / throughput benchmarking
# --------------------------------------------------------------------------


def measure_latency(predict_fn, inputs, warmup=5, runs=30, batch_size=1):
    """Run predict_fn(inputs) repeatedly and report wall-clock percentiles.

    Warm-up matters: the first calls pay for graph tracing, kernel autotuning
    and (on GPU) cuDNN algorithm selection. Reporting those numbers is the
    single most common benchmarking mistake in deployment work.
    """
    for _ in range(warmup):
        predict_fn(inputs)

    samples = []
    for _ in range(runs):
        t0 = time.perf_counter()
        predict_fn(inputs)
        samples.append((time.perf_counter() - t0) * 1000.0)

    samples = np.array(sorted(samples))
    p50 = float(np.percentile(samples, 50))
    return {
        "mean_ms": round(float(samples.mean()), 2),
        "p50_ms": round(p50, 2),
        "p90_ms": round(float(np.percentile(samples, 90)), 2),
        "p95_ms": round(float(np.percentile(samples, 95)), 2),
        "throughput_rps": round(batch_size / (p50 / 1000.0), 1),
    }


def device_label():
    return "GPU" if tf.config.list_physical_devices("GPU") else "CPU"


# --------------------------------------------------------------------------
# 4. The optimization ledger
# --------------------------------------------------------------------------


def _ledger_file():
    (_ROOT / "reports").mkdir(parents=True, exist_ok=True)
    return _ROOT / "reports" / "ledger.json"


def load_ledger():
    f = _ledger_file()
    if not f.exists():
        return []
    return json.loads(f.read_text())


def record(stage, **fields):
    """Insert or replace a ledger row. Stage names are unique keys, so
    re-running a cell updates the row instead of duplicating it."""
    ledger = [e for e in load_ledger() if e.get("stage") != stage]
    entry = {"stage": stage, "recorded_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    entry.update(fields)
    ledger.append(entry)
    _ledger_file().write_text(json.dumps(ledger, indent=2))
    return entry


def ledger_df(columns=None):
    import pandas as pd

    df = pd.DataFrame(load_ledger())
    if df.empty:
        return df
    preferred = [
        "stage",
        "model",
        "task",
        "dataset",
        "params_m",
        "size_mb",
        "quality",
        "quality_metric",
        "p50_ms",
        "p95_ms",
        "throughput_rps",
        "device",
        "notes",
    ]
    cols = columns or [c for c in preferred if c in df.columns]
    extra = [c for c in df.columns if c not in cols and c != "recorded_at"]
    return df[cols + extra]


# --------------------------------------------------------------------------
# 5. A small, explicit training loop
# --------------------------------------------------------------------------


def train(
    model,
    dataset,
    loss_fn,
    optimizer,
    epochs=1,
    steps_per_epoch=None,
    log_every=50,
    on_step_end=None,
    clip_norm=1.0,
):
    """Generic GradientTape loop.

    loss_fn(model, batch, training) -> scalar loss tensor.
    on_step_end(global_step) -> optional Python callback, used by the pruning
    lab to update sparsity masks between steps.
    """

    @tf.function
    def train_step(batch):
        with tf.GradientTape() as tape:
            loss = loss_fn(model, batch, True)
        grads = tape.gradient(loss, model.trainable_variables)
        pairs = [
            (g, v) for g, v in zip(grads, model.trainable_variables) if g is not None
        ]
        if clip_norm:
            gs, _ = tf.clip_by_global_norm([g for g, _ in pairs], clip_norm)
            pairs = list(zip(gs, [v for _, v in pairs]))
        optimizer.apply_gradients(pairs)
        return loss

    global_step = 0
    history = []
    for epoch in range(epochs):
        running, seen = 0.0, 0
        t0 = time.time()
        for step, batch in enumerate(dataset):
            loss = float(train_step(batch))
            running += loss
            seen += 1
            global_step += 1
            if on_step_end is not None:
                on_step_end(global_step)
            if log_every and global_step % log_every == 0:
                print(
                    f"  epoch {epoch + 1} | step {global_step:>5} | "
                    f"loss {running / seen:.4f}"
                )
                running, seen = 0.0, 0
            if steps_per_epoch and step + 1 >= steps_per_epoch:
                break
        history.append({"epoch": epoch + 1, "seconds": round(time.time() - t0, 1)})
        print(f"  epoch {epoch + 1} finished in {history[-1]['seconds']}s")
    return history


# --------------------------------------------------------------------------
# 6. Evaluation helpers
# --------------------------------------------------------------------------


def evaluate_accuracy(logits_fn, dataset):
    """logits_fn(features) -> array of shape [batch, num_classes]."""
    correct, total = 0, 0
    for features, labels in dataset:
        logits = np.asarray(logits_fn(features))
        preds = logits.argmax(axis=-1)
        labels = np.asarray(labels)
        correct += int((preds == labels).sum())
        total += int(labels.shape[0])
    return correct / max(total, 1)


def hf_logits_fn(model):
    """Wrap a Hugging Face TF model so it returns a plain logits tensor and is
    compiled once into a graph (fair, low-overhead benchmarking)."""

    @tf.function(reduce_retracing=True)
    def fn(features):
        return model(features, training=False).logits

    return fn


def banner(title):
    line = "=" * max(60, len(title) + 4)
    print(f"\n{line}\n  {title}\n{line}")
'''

from pathlib import Path
Path(ROOT, 'labkit.py').write_text(LABKIT_SRC)

import sys, importlib
sys.path.insert(0, ROOT)
import labkit as lk
importlib.reload(lk)
lk.set_root(ROOT)
lk.banner('lab kit ready')
print('root  :', lk.root())
print('device:', lk.device_label())
print('ledger rows:', [r['stage'] for r in lk.load_ledger()])

---
## 1. The arithmetic of quantization

### 1.1 Numeric formats

| Format | Bits | Range | Typical use |
|---|---|---|---|
| `float32` | 32 | ~1e±38, ~7 decimal digits | training default, the baseline |
| `float16` | 16 | ~1e±5, ~3 digits | GPU inference; can overflow |
| `bfloat16` | 16 | fp32's range, ~2 digits | training on TPU/modern GPU; rarely overflows |
| `int8` | 8 | −128 … 127 | CPU inference; needs scale + zero-point |
| `int4` | 4 | −8 … 7 | aggressive LLM weight-only quantization |

Note the difference between fp16 and bf16: bf16 keeps fp32's exponent and sacrifices mantissa bits, so it has the same dynamic range and is far less prone to overflow. fp16 has more precision within a much narrower range.

### 1.2 Affine quantization

To store a real tensor $x$ in $b$ bits we need a linear map from a real interval onto the integer grid:

$$q = \mathrm{clip}\!\left(\mathrm{round}\!\left(\frac{x}{s}\right) + z,\ q_{\min},\ q_{\max}\right), \qquad \hat{x} = s\,(q - z)$$

- $s$ (**scale**) is the width of one integer step, in real units.
- $z$ (**zero-point**) is the integer that represents exact 0. Choosing $s$ and $z$ so that real 0 maps exactly onto an integer matters: padding, ReLU outputs and masks are exactly zero, and a rounding error there propagates everywhere.

**Symmetric** quantization forces $z = 0$ and uses $s = \max|x| / q_{\max}$. It is cheaper (no zero-point correction term in the matmul) and it fits weights well, because weight distributions are roughly zero-centred. **Asymmetric** quantization fits $[x_{\min}, x_{\max}]$ and is the right choice for activations after a ReLU, which are one-sided.

$\hat{x} - x$ is the **quantization error**, and it has two sources: *rounding* (bounded by $s/2$) and *clipping* (unbounded, from values outside the chosen range). Rounding error is the price of the format. Clipping error is a choice - and outliers are what makes that choice hard.

### 1.3 Per-tensor vs per-channel

One scale for an entire weight matrix means a single outlier column drags the scale up and crushes the resolution of every other column. Per-channel quantization gives each output channel its own scale, at the cost of storing one extra float per channel - negligible.

For transformers this is not a marginal improvement. Attention and feed-forward projections routinely contain a handful of channels with much larger magnitudes than the rest; per-tensor int8 on those layers is a well-known way to lose several points of accuracy for no reason. You will measure this yourself in the next section.

### 1.4 Implement it, then look at real weights

The function below is the whole technique. Everything a production quantization toolkit does is a refinement of these six lines - better range selection, better granularity, error compensation.

In [ ]:
def quantize_dequantize(x, num_bits=8, axis=None, symmetric=True):
    # Affine quantize then dequantize. axis=None -> per-tensor.
    # axis=k -> a separate scale per slice along axis k (per-channel).
    x = tf.convert_to_tensor(x, tf.float32)
    qmin, qmax = -(2 ** (num_bits - 1)), 2 ** (num_bits - 1) - 1

    reduce_axes = None
    if axis is not None:
        reduce_axes = [i for i in range(len(x.shape)) if i != axis]

    if symmetric:
        amax = tf.reduce_max(tf.abs(x), axis=reduce_axes, keepdims=True)
        scale = tf.maximum(amax / qmax, 1e-12)
        zero = tf.zeros_like(scale)
    else:
        xmin = tf.reduce_min(x, axis=reduce_axes, keepdims=True)
        xmax = tf.reduce_max(x, axis=reduce_axes, keepdims=True)
        scale = tf.maximum((xmax - xmin) / (qmax - qmin), 1e-12)
        zero = tf.round(qmin - xmin / scale)

    q = tf.clip_by_value(tf.round(x / scale) + zero, qmin, qmax)
    return scale * (q - zero)


# Sanity check on a tensor whose exact quantization we can reason about.
demo = tf.constant([-1.0, -0.5, 0.0, 0.25, 1.0])
print("original   :", demo.numpy())
print("int8 (sym) :", quantize_dequantize(demo).numpy().round(5))
print("int4 (sym) :", quantize_dequantize(demo, num_bits=4).numpy().round(5))
print("int2 (sym) :", quantize_dequantize(demo, num_bits=2).numpy().round(5))

### 1.5 Load the Lab 1 model and quantize its real weights

If the next cell reports a missing model, go back and run Lab 1 Section 4 - it produces `models/teacher-bert-sst2/`.

In [ ]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

TEACHER_DIR = os.path.join(ROOT, "models", "teacher-bert-sst2")
if not os.path.isdir(TEACHER_DIR):
    raise FileNotFoundError(
        f"{TEACHER_DIR} not found. Run Lab 1 (Section 4) to produce the baseline model, "
        "or set USE_DRIVE=True in both notebooks so artifacts persist.")

tokenizer = AutoTokenizer.from_pretrained(TEACHER_DIR)
model = TFAutoModelForSequenceClassification.from_pretrained(TEACHER_DIR)
_ = model(model.dummy_inputs, training=False)
print(f"loaded baseline: {model.num_parameters()/1e6:.1f}M parameters")

In [ ]:
import pandas as pd

# Pick a representative feed-forward projection from the middle of the encoder.
candidates = [v for v in model.trainable_variables
              if "intermediate" in v.name and "kernel" in v.name and len(v.shape) == 2]
target = candidates[len(candidates) // 2]   # a mid-stack feed-forward projection
W = target.numpy()
print("inspecting", target.name, "shape", W.shape)

flat = np.abs(W).flatten()
print(f"weight |w|: median {np.median(flat):.4f} | p99 {np.percentile(flat, 99):.4f} "
      f"| max {flat.max():.4f}  -> the max is {flat.max()/np.median(flat):.0f}x the median")

rows = []
for bits in (8, 6, 4):
    for label, axis in (("per-tensor", None), ("per-channel", 1)):
        Wq = quantize_dequantize(W, num_bits=bits, axis=axis).numpy()
        err = Wq - W
        rows.append({
            "bits": bits, "granularity": label,
            "mean_abs_err": float(np.abs(err).mean()),
            "max_abs_err": float(np.abs(err).max()),
            "rel_frobenius_err": float(np.linalg.norm(err) / np.linalg.norm(W)),
        })
pd.DataFrame(rows)

Two things to read off that table:

1. **Per-channel beats per-tensor at every bit width**, and the gap widens as bits shrink. The outlier ratio printed above is the reason - one extreme column sets the scale for all of them under per-tensor.
2. **Error roughly doubles per bit removed**, exactly as $s = \max|x|/q_{\max}$ predicts. int8 is nearly free; int4 needs help (grouped scales, error compensation, or quantization-aware training).

Relative Frobenius error is a proxy, not a verdict. Weight error only matters through its effect on the output - so we now measure end-task accuracy.

---
## 2. Quantize the real model and test it under distribution shift

### 2.1 The IMDB twist

The syllabus assigns IMDB to this lab, and there is a better use for it than training yet another sentiment model. Our Lab 1 baseline was trained on **SST-2**: short fragments, a median of around 9 words. **IMDB** is the same binary sentiment task on full-length reviews - hundreds of words, different vocabulary, different syntax.

Evaluating our quantized model on IMDB therefore answers a question that an in-distribution test cannot: **does quantization damage the model more when the input distribution shifts?** This matters in production because your calibration data is always drawn from what you have, and your traffic is always slightly different. A quantized model that only holds up in-distribution is a model that will degrade quietly after release.

In [ ]:
from datasets import load_dataset

MAX_LEN = 128
N_IMDB  = 2_000

imdb_raw = load_dataset("stanfordnlp/imdb")["test"].shuffle(seed=42).select(range(N_IMDB))
sst2_val = load_dataset("nyu-mll/glue", "sst2")["validation"]

def encode(texts):
    enc = tokenizer(list(texts), max_length=MAX_LEN, truncation=True,
                    padding="max_length", return_tensors="np")
    return {k: np.asarray(v, np.int32) for k, v in enc.items()}

def make_ds(texts, labels, bs=64):
    return (tf.data.Dataset.from_tensor_slices((encode(texts), np.asarray(labels, np.int32)))
            .batch(bs).prefetch(tf.data.AUTOTUNE))

imdb_ds = make_ds(imdb_raw["text"], imdb_raw["label"])
sst2_ds = make_ds(sst2_val["sentence"], sst2_val["label"])

sst_len  = np.median([len(t.split()) for t in sst2_val["sentence"]])
imdb_len = np.median([len(t.split()) for t in imdb_raw["text"]])
print(f"median length - SST-2 {sst_len:.0f} words | IMDB {imdb_len:.0f} words")
print(f"at MAX_LEN={MAX_LEN}, roughly "
      f"{np.mean([len(t.split()) > MAX_LEN for t in imdb_raw['text']])*100:.0f}% "
      f"of IMDB reviews are truncated")

In [ ]:
logits_fn = lk.hf_logits_fn(model)

fp32_sst2 = lk.evaluate_accuracy(logits_fn, sst2_ds)
fp32_imdb = lk.evaluate_accuracy(logits_fn, imdb_ds)
print(f"fp32 baseline | SST-2 {fp32_sst2:.4f} | IMDB {fp32_imdb:.4f}")
print("The IMDB gap is the cost of distribution shift and truncation - "
      "it is NOT a quantization effect. It is our new reference line.")

### 2.2 Apply simulated quantization to the whole encoder

We quantize-dequantize every 2-D weight matrix in place, keeping the originals so we can restore fp32 and compare variants.

What we quantize and what we leave alone is a real engineering decision:

- **2-D kernels** (attention Q/K/V/output projections, feed-forward) - quantized. These are the parameter mass and the FLOP mass.
- **Biases and LayerNorm parameters** - left in fp32. They are a rounding error's worth of memory and are numerically sensitive.
- **The embedding table** - optional, and controlled by a flag below. It is nearly a quarter of BERT-base's parameters, so quantizing it is a large size win, but it is a lookup table rather than a matmul operand and errors there enter at layer zero and compound through the whole stack.

In [ ]:
ORIGINALS = {v.name: v.numpy().copy() for v in model.weights}

def restore_fp32():
    for v in model.weights:
        v.assign(ORIGINALS[v.name])

def apply_weight_quantization(num_bits=8, per_channel=True, include_embeddings=False):
    restore_fp32()
    touched = quantized_params = 0
    for v in model.weights:
        name = v.name.lower()
        is_kernel = ("kernel" in name) and len(v.shape) == 2
        is_embed  = "embeddings" in name and len(v.shape) == 2
        if not (is_kernel or (include_embeddings and is_embed)):
            continue
        axis = 1 if (per_channel and is_kernel) else None
        v.assign(quantize_dequantize(v, num_bits=num_bits, axis=axis))
        touched += 1
        quantized_params += int(np.prod(v.shape))
    return touched, quantized_params

n_layers, n_params = apply_weight_quantization(8, True, False)
print(f"quantized {n_layers} weight matrices covering {n_params/1e6:.1f}M parameters")

In [ ]:
results = []

def evaluate_variant(label, **kwargs):
    n_l, n_p = apply_weight_quantization(**kwargs)
    fn = lk.hf_logits_fn(model)
    row = {
        "variant": label,
        "quantized_params_m": round(n_p / 1e6, 1),
        "sst2_acc": round(lk.evaluate_accuracy(fn, sst2_ds), 4),
        "imdb_acc": round(lk.evaluate_accuracy(fn, imdb_ds), 4),
    }
    row["sst2_delta"] = round(row["sst2_acc"] - fp32_sst2, 4)
    row["imdb_delta"] = round(row["imdb_acc"] - fp32_imdb, 4)
    results.append(row)
    print(f"  {label:<34} SST-2 {row['sst2_acc']:.4f} ({row['sst2_delta']:+.4f})  "
          f"IMDB {row['imdb_acc']:.4f} ({row['imdb_delta']:+.4f})")
    return row

lk.banner("simulated weight quantization sweep")
evaluate_variant("int8 per-channel",              num_bits=8, per_channel=True)
evaluate_variant("int8 per-tensor",               num_bits=8, per_channel=False)
evaluate_variant("int8 per-channel + embeddings", num_bits=8, per_channel=True,
                 include_embeddings=True)
evaluate_variant("int4 per-channel",              num_bits=4, per_channel=True)
evaluate_variant("int4 per-tensor",               num_bits=4, per_channel=False)

restore_fp32()
pd.DataFrame(results)

### 2.3 What the sweep tells you

Read your own numbers against these expectations, and treat a large deviation as a bug to investigate rather than a curiosity:

- **int8 per-channel should be nearly free** - typically within a few tenths of a percentage point on both sets. This is the result that justifies quantization as a default, and it matches the sub-1% F1 impact reported in the case study.
- **int8 per-tensor should be measurably worse.** Same bit width, same memory, worse accuracy - the difference is purely the granularity choice. If someone reports that int8 "does not work" for their transformer, this is the first thing to check.
- **Quantizing embeddings buys a large size reduction** for a small extra accuracy cost. Whether that is a good trade depends on whether you are memory-bound or accuracy-bound.
- **int4 falls off a cliff**, and per-tensor int4 usually falls further. Getting int4 to work requires techniques beyond round-to-nearest - group-wise scales, or quantization-aware training as in Section 4.

Now check the deltas in the two columns against each other. If `imdb_delta` is consistently more negative than `sst2_delta`, quantization is hurting more under distribution shift, and your calibration and acceptance testing need to cover the traffic you actually expect - not just your validation split.

### 2.4 The size that quantization actually buys

Simulated quantization proves the accuracy claim; it does not shrink anything, because the values still sit in fp32 containers. To measure the real size effect we serialise the weights in each format - which is precisely what a serving runtime stores.

In [ ]:
kernels = [v for v in model.weights if "kernel" in v.name.lower() and len(v.shape) == 2]
embeds  = [v for v in model.weights if "embeddings" in v.name.lower() and len(v.shape) == 2]
# Compare by name: `v in list_of_variables` would do elementwise tensor comparison.
_grouped = {v.name for v in kernels} | {v.name for v in embeds}
others  = [v for v in model.weights if v.name not in _grouped]

def bytes_for(vars_, bytes_per_param, scale_overhead=0):
    total = sum(int(np.prod(v.shape)) * bytes_per_param for v in vars_)
    return total + scale_overhead * 4

k_params = sum(int(np.prod(v.shape)) for v in kernels)
e_params = sum(int(np.prod(v.shape)) for v in embeds)
o_params = sum(int(np.prod(v.shape)) for v in others)
n_scales = sum(v.shape[1] for v in kernels)          # one fp32 scale per output channel

plans = {
    "fp32 (baseline)":        bytes_for(kernels, 4) + bytes_for(embeds, 4) + bytes_for(others, 4),
    "fp16 everything":        bytes_for(kernels, 2) + bytes_for(embeds, 2) + bytes_for(others, 2),
    "int8 kernels only":      bytes_for(kernels, 1, n_scales) + bytes_for(embeds, 4) + bytes_for(others, 4),
    "int8 kernels + embeds":  bytes_for(kernels, 1, n_scales) + bytes_for(embeds, 1) + bytes_for(others, 4),
}
size_df = pd.DataFrame([
    {"plan": k, "size_mb": round(v / 1e6, 1), "vs_fp32": round(v / plans["fp32 (baseline)"], 3)}
    for k, v in plans.items()])

print(f"kernels {k_params/1e6:.1f}M | embeddings {e_params/1e6:.1f}M | "
      f"biases+norms {o_params/1e6:.2f}M params")
size_df

In [ ]:
# Verify the accounting against real files rather than trusting the arithmetic.
probe_dir = os.path.join(ROOT, "data", "size_probe")
os.makedirs(probe_dir, exist_ok=True)
arrays = {v.name.replace("/", "_"): v.numpy() for v in kernels[:24]}

np.savez(os.path.join(probe_dir, "fp32.npz"), **arrays)
np.savez(os.path.join(probe_dir, "fp16.npz"), **{k: v.astype(np.float16) for k, v in arrays.items()})
int8 = {}
for k, v in arrays.items():
    scale = np.maximum(np.abs(v).max(axis=0, keepdims=True) / 127.0, 1e-12)
    int8[k] = np.clip(np.round(v / scale), -128, 127).astype(np.int8)
    int8[k + "__scale"] = scale.astype(np.float32)
np.savez(os.path.join(probe_dir, "int8.npz"), **int8)

for fmt in ("fp32", "fp16", "int8"):
    print(f"{fmt}: {lk.size_mb(os.path.join(probe_dir, fmt + '.npz')):6.2f} MB "
          f"(24 kernel matrices)")

In [ ]:
# Latency is unchanged by simulated quantization - the ops are still fp32.
# We record it so the ledger stays honest about which effects were measured how.
val_feats = encode(sst2_val["sentence"])
bs1 = {k: tf.constant(v[:1]) for k, v in val_feats.items()}

apply_weight_quantization(8, per_channel=True)
q_fn = lk.hf_logits_fn(model)
q_acc_sst2 = lk.evaluate_accuracy(q_fn, sst2_ds)
q_acc_imdb = lk.evaluate_accuracy(q_fn, imdb_ds)
q_lat = lk.measure_latency(q_fn, bs1, warmup=10, runs=50, batch_size=1)
restore_fp32()

int8_size = plans["int8 kernels only"] / 1e6
fp16_size = plans["fp16 everything"] / 1e6

lk.record("quantized-int8", model="BERT-base, int8 per-channel kernels",
          task="binary sentiment", dataset="SST-2 / IMDB",
          params_m=round(model.num_parameters()/1e6, 1), size_mb=round(int8_size, 1),
          quality=round(q_acc_sst2, 4), quality_metric="accuracy",
          p50_ms=q_lat["p50_ms"], p95_ms=q_lat["p95_ms"],
          throughput_rps=q_lat["throughput_rps"], device=lk.device_label(),
          imdb_accuracy=round(q_acc_imdb, 4),
          notes="size projected from format; accuracy measured; latency unchanged (simulated)")

lk.record("quantized-fp16", model="BERT-base, fp16 weights",
          task="binary sentiment", dataset="SST-2 / IMDB",
          params_m=round(model.num_parameters()/1e6, 1), size_mb=round(fp16_size, 1),
          quality=round(lk.evaluate_accuracy(
              lk.hf_logits_fn(model), sst2_ds), 4), quality_metric="accuracy",
          p50_ms=q_lat["p50_ms"], p95_ms=q_lat["p95_ms"],
          throughput_rps=q_lat["throughput_rps"], device=lk.device_label(),
          notes="fp16 is typically lossless for BERT-scale inference")

lk.ledger_df()

---
## 3. Real quantization end-to-end with TFLite

Now we measure the effects simulation cannot: **actual bytes on disk and actual milliseconds on CPU**. TFLite converts small, statically-shaped Keras models reliably, so we build one on IMDB and take it all the way through.

The model is a bag-of-words classifier: a multi-hot vector over the top vocabulary terms, into two dense layers. It is not state of the art and it is not meant to be - it is a model that every quantization path in the toolkit supports, so nothing in this section fails for reasons unrelated to quantization.

### 3.1 The three post-training quantization modes

| Mode | Weights | Activations | Needs calibration data? | Where it runs well |
|---|---|---|---|---|
| **Dynamic range** | int8 | quantized on the fly, per batch | no | CPU; the standard default |
| **Float16** | fp16 | fp32 | no | GPU/accelerator with fp16 support |
| **Full integer** | int8 | int8 (fixed ranges) | **yes** | integer-only hardware, edge, microcontrollers |

Dynamic range is what the case study used, and it is the right first choice: no calibration data, no pipeline changes, and the biggest CPU win.

Full-integer quantization needs a **representative dataset** - a few hundred real inputs used to observe activation ranges. Two rules: it must come from the same distribution as production traffic, and it must include the tails. A calibration set of only typical inputs produces ranges that clip on the unusual ones, which is a bug you will only see in production.

In [ ]:
VOCAB, N_TRAIN_BOW, N_TEST_BOW = 4_000, 15_000, 5_000

(x_tr, y_tr), (x_te, y_te) = tf.keras.datasets.imdb.load_data(num_words=VOCAB)

def multi_hot(sequences, dim):
    out = np.zeros((len(sequences), dim), dtype=np.float32)
    for i, seq in enumerate(sequences):
        out[i, [t for t in seq if t < dim]] = 1.0
    return out

Xtr = multi_hot(x_tr[:N_TRAIN_BOW], VOCAB); Ytr = np.asarray(y_tr[:N_TRAIN_BOW], np.int32)
Xte = multi_hot(x_te[:N_TEST_BOW],  VOCAB); Yte = np.asarray(y_te[:N_TEST_BOW],  np.int32)
print("train", Xtr.shape, "test", Xte.shape, f"({Xtr.nbytes/1e6:.0f} MB train features)")

bow = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(VOCAB,), name="tokens"),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(2),
], name="imdb_bow")
bow.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
            loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
            metrics=["accuracy"])
bow.fit(Xtr, Ytr, epochs=4, batch_size=128, validation_split=0.1, verbose=2)
bow_acc = bow.evaluate(Xte, Yte, verbose=0)[1]
print(f"\nfp32 Keras accuracy on IMDB test: {bow_acc:.4f}")

In [ ]:
TFLITE_DIR = os.path.join(ROOT, "models", "tflite")
os.makedirs(TFLITE_DIR, exist_ok=True)

def representative_dataset():
    # 300 real inputs, one at a time, so the converter can observe
    # activation ranges at every layer.
    for i in range(300):
        yield [Xtr[i:i + 1]]

def convert(name, configure):
    converter = tf.lite.TFLiteConverter.from_keras_model(bow)
    configure(converter)
    blob = converter.convert()
    path = os.path.join(TFLITE_DIR, f"{name}.tflite")
    with open(path, "wb") as f:
        f.write(blob)
    return path

def cfg_float32(c):
    pass

def cfg_dynamic(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]

def cfg_fp16(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.target_spec.supported_types = [tf.float16]

def cfg_int8(c):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = representative_dataset
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    c.inference_input_type = tf.int8
    c.inference_output_type = tf.int8

paths = {}
for name, cfg in [("float32", cfg_float32), ("dynamic_int8", cfg_dynamic),
                  ("float16", cfg_fp16), ("full_int8", cfg_int8)]:
    try:
        paths[name] = convert(name, cfg)
        print(f"{name:<14} -> {lk.size_mb(paths[name]):6.3f} MB")
    except Exception as e:
        print(f"{name:<14} -> conversion failed: {type(e).__name__}: {e}")

### 3.2 Evaluate the converted models on CPU

TFLite's interpreter is a CPU runtime, so these numbers are directly comparable to a CPU serving instance - the deployment target the case study moved to.

The full-integer model takes and returns **int8 tensors**, so we have to quantize the input and dequantize the output ourselves using the scale and zero-point the converter recorded. That plumbing is not incidental: it is exactly the code you write in your Flask handler on Day 2 when the serving runtime expects integers.

In [ ]:
def tflite_evaluate(path, X, Y, n_latency=200):
    interp = tf.lite.Interpreter(model_path=path, num_threads=1)
    interp.allocate_tensors()
    inp, out = interp.get_input_details()[0], interp.get_output_details()[0]

    def prep(batch):
        if inp["dtype"] == np.int8:
            scale, zero = inp["quantization"]
            return np.clip(np.round(batch / scale) + zero, -128, 127).astype(np.int8)
        return batch.astype(inp["dtype"])

    def read(raw):
        if out["dtype"] == np.int8:
            scale, zero = out["quantization"]
            return (raw.astype(np.float32) - zero) * scale
        return raw

    correct = 0
    for i in range(len(X)):
        interp.set_tensor(inp["index"], prep(X[i:i + 1]))
        interp.invoke()
        logits = read(interp.get_tensor(out["index"]))
        correct += int(logits.argmax() == Y[i])

    sample = prep(X[:1])
    for _ in range(20):
        interp.set_tensor(inp["index"], sample); interp.invoke()
    times = []
    for _ in range(n_latency):
        t0 = time.perf_counter()
        interp.set_tensor(inp["index"], sample); interp.invoke()
        times.append((time.perf_counter() - t0) * 1000)

    return {"accuracy": round(correct / len(X), 4),
            "p50_ms": round(float(np.percentile(times, 50)), 4),
            "p95_ms": round(float(np.percentile(times, 95)), 4),
            "size_mb": round(lk.size_mb(path), 4)}

lk.banner("TFLite post-training quantization on CPU")
tflite_rows = []
for name, path in paths.items():
    r = tflite_evaluate(path, Xte, Yte)
    r["variant"] = name
    tflite_rows.append(r)
    print(f"  {name:<14} acc {r['accuracy']:.4f} | {r['size_mb']:.3f} MB | "
          f"p50 {r['p50_ms']:.3f} ms")

tfl = pd.DataFrame(tflite_rows)[["variant", "size_mb", "accuracy", "p50_ms", "p95_ms"]]
base = tfl[tfl.variant == "float32"].iloc[0]
tfl["size_vs_fp32"] = (tfl.size_mb / base.size_mb).round(3)
tfl["speedup"] = (base.p50_ms / tfl.p50_ms).round(2)
tfl["acc_delta"] = (tfl.accuracy - base.accuracy).round(4)
tfl

Now you have the honest picture, measured rather than projected: roughly 4× smaller for dynamic-range and full-integer int8, roughly 2× for float16, and an accuracy delta in the third decimal place.

The speedup column deserves a caveat. On a small dense model, per-invocation overhead is a large share of the total, so the int8 speedup here will look modest compared to the case study's. Speedup from quantization scales with how matmul-bound the model is - which is why a BERT-sized model on CPU sees a far larger gain than this bag-of-words classifier does.

---
## 4. Quantization-aware training

Post-training quantization rounds a model that was trained assuming infinite precision. **Quantization-aware training** instead inserts fake-quantize operations into the forward pass during training, so the weights adapt to the grid they will eventually be rounded onto.

The mechanism worth understanding is the **straight-through estimator**. `round()` has zero gradient almost everywhere, which would block learning entirely. QAT therefore quantizes in the forward pass and passes the gradient through unchanged in the backward pass - an approximation that is wrong in the small and right in the aggregate.

When to reach for QAT:

- PTQ costs more accuracy than your budget allows.
- You are going below 8 bits, where PTQ reliably breaks down.
- Your activations have long tails that no static calibration range handles well.

The cost is a training run and a more complex pipeline, so PTQ first, QAT only if PTQ fails your acceptance test.

In [ ]:
import tensorflow_model_optimization as tfmot

qat_model = tfmot.quantization.keras.quantize_model(bow)
qat_model.compile(optimizer=tf.keras.optimizers.Adam(2e-4),
                  loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  metrics=["accuracy"])
print("fake-quant layers inserted:")
for layer in qat_model.layers[:6]:
    print("  ", layer.__class__.__name__)

# A short fine-tune is enough: the weights only need to settle onto the grid.
qat_model.fit(Xtr, Ytr, epochs=2, batch_size=128, validation_split=0.1, verbose=2)
print(f"\nQAT model accuracy (still simulated, fp32 container): "
      f"{qat_model.evaluate(Xte, Yte, verbose=0)[1]:.4f}")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
qat_path = os.path.join(TFLITE_DIR, "qat_int8.tflite")
with open(qat_path, "wb") as f:
    f.write(converter.convert())

qat_result = tflite_evaluate(qat_path, Xte, Yte)
ptq_result = next(r for r in tflite_rows if r["variant"] == "full_int8")

print(f"fp32 baseline    accuracy {base.accuracy:.4f}")
print(f"PTQ full int8    accuracy {ptq_result['accuracy']:.4f} "
      f"({ptq_result['accuracy'] - base.accuracy:+.4f})")
print(f"QAT int8         accuracy {qat_result['accuracy']:.4f} "
      f"({qat_result['accuracy'] - base.accuracy:+.4f})")
print(f"\nsizes: PTQ {ptq_result['size_mb']:.3f} MB | QAT {qat_result['size_mb']:.3f} MB")

At 8 bits on a well-conditioned model, QAT and PTQ often land within noise of each other - and that is a useful result, not a disappointing one. **It tells you PTQ was sufficient, and you can skip the training pipeline.** The place QAT earns its cost is 4 bits and below, or models whose PTQ accuracy drop is unacceptable. Exercise 3 asks you to find that crossover on this model.

---
## 5. Exercises

**1. Find the bit floor.**
Sweep `num_bits` from 8 down to 2 with per-channel quantization on the BERT model and plot SST-2 accuracy against bits. Where is the knee? Repeat with embeddings included - does the knee move?

**2. Sensitivity analysis.**
Modify `apply_weight_quantization` to quantize only one encoder layer at a time and record the accuracy drop. Which layers are most sensitive? Use the result to design a **mixed-precision** plan - int8 for most layers, fp16 for the two most sensitive - and measure the accuracy/size trade-off against uniform int8.

**3. Push QAT below int8.**
Compare PTQ and QAT at 4 bits on the bag-of-words model using `tfmot`'s quantization schemes. At what bit width does QAT's advantage become decisive?

**4. Break the calibration set.**
Rebuild `representative_dataset` from only the 300 *shortest* IMDB reviews and re-convert the full-integer model. Measure accuracy on the full test set. How much accuracy did a bad calibration set cost, and what does that imply for how you would sample calibration data from production traffic?

**5. Bridge to Day 2.**
The full-integer TFLite model needs input quantization and output dequantization in the request handler. Write the two functions you would put in a Flask endpoint, using `interp.get_input_details()[0]["quantization"]`. Keep them - Day 2 will use this exact pattern.

---
## 6. Wrap-up

### What you measured

| Claim | How it was established |
|---|---|
| int8 per-channel costs almost no accuracy | simulated quantization on the real BERT model, on two datasets |
| per-channel is strictly better than per-tensor | error analysis plus end-task accuracy |
| int8 is ~4× smaller, fp16 ~2× | format arithmetic, verified against serialised files, and confirmed end-to-end in TFLite |
| int8 is faster on CPU | measured with the TFLite interpreter |
| QAT ≈ PTQ at 8 bits | converted both and evaluated |

### Carry forward

1. **Per-channel, symmetric, int8 weights is the default.** Deviate only with evidence.
2. **Simulation answers "does quality survive". A runtime answers "is it smaller and faster".** Never quote one as if it were the other.
3. **Calibration data is part of the model.** Treat it as an artifact with provenance, not a convenience sample.
4. **Test under distribution shift.** Your validation split flatters you; your traffic will not.

### Checkpoint

- [ ] The ledger contains `quantized-int8` and `quantized-fp16`.
- [ ] You can state your model's int8 accuracy delta on SST-2 *and* IMDB.
- [ ] You have four `.tflite` files under `models/tflite/` with measured sizes.

### Next

**Lab 4 - Model Pruning.** The third lever: removing weights entirely. Per the syllabus we work on SST-2. You will find pruning is the least reliable of the three on transformers - and understanding exactly *why* unstructured sparsity often fails to speed anything up is the point of the lab.